In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
!pip install wandb --quiet

## Config

Set `REPO_ROOT` to the repo root on your Google Drive. All other paths are relative to it.

In [ ]:
import os
import sys
import types
import subprocess

import yaml
import torch
import wandb

In [ ]:
REPO_ROOT = "/content/drive/MyDrive/Rutishauser_Lab/AI4Sci/METHODS/DiffusionBANMap/"
CONFIG_FILE = "config/diffusion_full_imagenet64_flow.yaml"
RUN_NAME = "run2"
RUN_NOTES = "Conditional on AlexNet fc6 latents (PCA down to 50 dim), ImageNet64, flow matching"
RESUME_CKPT_PATH = None

## Imports

In [ ]:
sys.path.insert(0, REPO_ROOT)

import importlib.metadata
if not hasattr(importlib.metadata, '_patched'):

    _original_version = importlib.metadata.version
    def _patched_version(name):
        try:
            return _original_version(name)
        except importlib.metadata.PackageNotFoundError:
            return 'unknown'
    importlib.metadata.version = _patched_version
    importlib.metadata._patched = True

from data_utils import build_dataloaders
from models.beta_vae import BetaVAE
from models.unet import UNet
from scripts.diffusion_trainer import DiffusionTrainer
from utils import discover_device, count_model_params

## Load config & build UNet

In [ ]:
from utils import make_run_tag

config_path = os.path.join(REPO_ROOT, CONFIG_FILE)
with open(config_path) as f:
    cfg = types.SimpleNamespace(**yaml.safe_load(f))

cfg.output_dir = os.path.join(REPO_ROOT, f"{cfg.output_dir.lstrip('./')}_{make_run_tag(cfg)}")

# Only present for conditional models; unconditional configs leave these commented out
if hasattr(cfg, 'betavae_config_path'):
    cfg.betavae_config_path = os.path.join(REPO_ROOT, cfg.betavae_config_path.lstrip("./"))
if hasattr(cfg, 'betavae_ckpt_path'):
    cfg.betavae_ckpt_path = os.path.join(REPO_ROOT, cfg.betavae_ckpt_path.lstrip("./"))

if not os.path.isabs(cfg.data_dir):
    cfg.data_dir = os.path.join(REPO_ROOT, cfg.data_dir.lstrip("./"))

if cfg.unconditional:
    print("Train for unconditional generation")

In [ ]:
device = torch.device(discover_device())
print(f"Device: {device}")

model = UNet(
    in_channels=cfg.in_channels,
    image_size=cfg.image_size,
    model_channels=cfg.model_channels,
    channel_mult=cfg.channel_mult,
    num_res_blocks=cfg.num_res_blocks,
    attention_resolutions=cfg.attention_resolutions,
    dropout=cfg.dropout,
    latent_dim=cfg.latent_dim,
)
count_model_params(model)

## Load frozen β-VAE

In [ ]:
if cfg.unconditional:
    vae = None
    print("Unconditional — no VAE, z is a zero vector")
elif hasattr(cfg, 'betavae_config_path'):
    with open(cfg.betavae_config_path) as f:
        vae_cfg = types.SimpleNamespace(**yaml.safe_load(f))

    vae = BetaVAE(
        input_channels=vae_cfg.input_channels,
        input_height=vae_cfg.input_height,
        input_width=vae_cfg.input_width,
        latent_dim=vae_cfg.latent_dim,
        hidden_dim=vae_cfg.hidden_dim,
        beta=vae_cfg.beta,
        device=device,
    )
    ckpt = torch.load(cfg.betavae_ckpt_path, map_location=device)
    vae.load_state_dict(ckpt["model_state_dict"])
    vae.eval()
    print(f"Loaded β-VAE from step {ckpt['step']}")
else:
    # Conditional, but no VAE configured — the dataloader is expected to already provide
    # precomputed latents (e.g. AlexNet-fc6-PCA, see notebooks/alexnet_pca_latents.ipynb)
    # as (image, latent) pairs, so DiffusionTrainer never needs to call vae_model.encode().
    vae = None
    print("Conditional on precomputed latents — no VAE to load, expecting (image, latent) pairs from the dataloader")

## Data

In [ ]:
if cfg.dataset_type == "tiny_imagenet":
    subprocess.run(["wget", "http://cs231n.stanford.edu/tiny-imagenet-200.zip"], check=True)
    subprocess.run(["unzip", "-q", "tiny-imagenet-200.zip"], check=True)
    subprocess.run(["rm", "tiny-imagenet-200.zip"], check=True)
    subprocess.run(["ls", "/content/tiny-imagenet-200"], check=True)
elif cfg.dataset_type == "imagenet64":
    # Prepare train and val images
    subprocess.run([
        "python", os.path.join(REPO_ROOT, "scripts/prepare_imagenet64_dataset.py"),
        "--drive_train_part1_zip", os.path.join(REPO_ROOT, "data/imagenet64/Imagenet64_train_part1.zip"),
        "--drive_train_part2_zip", os.path.join(REPO_ROOT, "data/imagenet64/Imagenet64_train_part2.zip"),
        "--drive_val_zip", os.path.join(REPO_ROOT, "data/imagenet64/Imagenet64_val.zip"),
        "--data_dir", cfg.data_dir,
    ], check=True)

    # Only stage precomputed latents (e.g. AlexNet-fc6-PCA) for runs that are actually
    # conditioned on them — same signal as the VAE-loading cell above, so an unconditional
    # or VAE-conditioned imagenet64 run never accidentally picks up (image, latent) pairs.
    if not cfg.unconditional and not hasattr(cfg, 'betavae_config_path'):
        subprocess.run([
            "cp", os.path.join(REPO_ROOT, f"data/imagenet64_alexnet_fc6_pca_latents/train_latents_z{cfg.latent_dim}.npy"),
            cfg.data_dir,
        ], check=True)
        subprocess.run([
            "cp", os.path.join(REPO_ROOT, f"data/imagenet64_alexnet_fc6_pca_latents/val_latents_z{cfg.latent_dim}.npy"),
            cfg.data_dir,
        ], check=True)
        print("Staged precomputed latents into data_dir — conditioning on precomputed latents")
    else:
        print("No precomputed-latents staging needed (unconditional or VAE-conditioned run)")

In [ ]:
train_dataloader, val_dataloader = build_dataloaders(cfg)

## Wandb

In [ ]:
key="..."
wandb.login(key=key)

wandb.init(
    project="diffusion_ban_map",
    name=f"{cfg.experiment_name}_{RUN_NAME}",
    notes=RUN_NOTES,
    config=vars(cfg),
)

## Train

In [ ]:
optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)

trainer = DiffusionTrainer(
    model=model,
    vae_model=vae,
    optimizer=optimizer,
    scheduler=None,
    train_dataloader=train_dataloader,
    val_dataloader=val_dataloader,
    config=cfg,
    device=device,
)
trainer.train(resume_ckpt_path=RESUME_CKPT_PATH)